# Connectome-Based Predictive Modeling (CPM) for Trait Anxiety

**Purpose:** This notebook gives you a working CPM pipeline you can run on Ubuntu in VS Code.

It starts with **simulated fMRI connectome data** so the whole method runs immediately. Then it shows exactly where to replace the simulated data with real subject connectomes from HCP / OpenNeuro / UK Biobank and a trait anxiety variable such as STAI-T.

CPM is machine learning in a relatively simple and interpretable form:

1. Each subject has a functional connectivity matrix.
2. Each subject has a behavioral score, here trait anxiety.
3. The model learns which brain connections predict anxiety in training subjects.
4. It predicts anxiety in held-out subjects.
5. We compare predicted vs actual anxiety.

**Important:** With only 24 subjects, this is a demonstration / pilot workflow, not a final strong publication-level model. But it is exactly the right way to learn CPM.

In [1]:
from pathlib import Path

# Project root (auto-detect)
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PHENO_DIR = DATA_DIR / "phenotypes"

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

# Create output folders if they don’t exist
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /home/john-walkey/Data/Research/connectome


## 0. Install packages

Run this in the VS Code terminal if needed:

```bash
python -m venv .venv
source .venv/bin/activate
pip install numpy pandas scipy scikit-learn matplotlib seaborn jupyter ipykernel
python -m ipykernel install --user --name cpm-env --display-name "Python (CPM)"
```

Then choose the **Python (CPM)** kernel in VS Code.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.model_selection import KFold, LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
print("Imports complete.")

Imports complete.


## 1. Create simulated CPM data

This creates 24 subjects. Each subject has a 20 × 20 functional connectivity matrix.

In real fMRI work, these matrices would come from ROI time-series correlations, for example:

- atlas ROI signals extracted with Nilearn
- correlation matrix computed per subject
- upper triangle used as CPM features

Here we simulate several hidden anxiety-related edges so the notebook can prove the method works.

In [3]:
# -----------------------------
# Simulation settings
# -----------------------------
n_subjects = 24
n_rois = 20

# Example ROI names. Replace these with atlas labels later.
roi_names = [
    "L_Insula", "R_Insula",
    "mPFC_DMN", "PCC_DMN", "L_AG_DMN", "R_AG_DMN",
    "dACC_SN", "L_FPN", "R_FPN"
] + [f"ROI_{i}" for i in range(10, n_rois + 1)]

# Trait anxiety scores: roughly STAI-T-like range
anxiety = np.random.normal(loc=45, scale=10, size=n_subjects)
anxiety = np.clip(anxiety, 20, 80)

# Random symmetric connectivity matrices
connectomes = []
for s in range(n_subjects):
    mat = np.random.normal(0, 0.15, size=(n_rois, n_rois))
    mat = (mat + mat.T) / 2
    np.fill_diagonal(mat, 1.0)
    connectomes.append(mat)
connectomes = np.array(connectomes)

# Add signal: some edges become stronger/weaker with anxiety
# These are artificial effects for learning CPM.
positive_edges = [(0, 2), (0, 3), (1, 3), (1, 4)]  # insula-DMN-ish
negative_edges = [(6, 7), (6, 8)]                  # SN-FPN-ish

z_anx = (anxiety - anxiety.mean()) / anxiety.std()
for s in range(n_subjects):
    for i, j in positive_edges:
        connectomes[s, i, j] += 0.25 * z_anx[s]
        connectomes[s, j, i] = connectomes[s, i, j]
    for i, j in negative_edges:
        connectomes[s, i, j] -= 0.20 * z_anx[s]
        connectomes[s, j, i] = connectomes[s, i, j]

subjects = pd.DataFrame({
    "subject_id": [f"sub-{i+1:02d}" for i in range(n_subjects)],
    "trait_anxiety": anxiety
})
subjects.head()

,subject_id,trait_anxiety
0,sub-01,49.967142
1,sub-02,43.617357
2,sub-03,51.476885
3,sub-04,60.230299
4,sub-05,42.658466


## 2. Convert each connectome into CPM features

A connectivity matrix is symmetric, so we only use the upper triangle.

For 20 ROIs, the number of possible unique edges is:


$$
\frac{20 \times 19}{2} = 190
$$

Each subject becomes one row. Each edge becomes one feature column.

In [4]:
def vectorize_connectomes(connectomes):
    """Return upper-triangle edge features from subject x ROI x ROI connectomes."""
    n_subjects, n_rois, _ = connectomes.shape
    triu_idx = np.triu_indices(n_rois, k=1)
    X = connectomes[:, triu_idx[0], triu_idx[1]]
    return X, triu_idx

X, triu_idx = vectorize_connectomes(connectomes)
y = subjects["trait_anxiety"].values

print("Feature matrix shape:", X.shape)
print("Number of subjects:", X.shape[0])
print("Number of edges/features:", X.shape[1])

Feature matrix shape: (24, 190)
Number of subjects: 24
Number of edges/features: 190


## 3. CPM training function

Inside each training fold, CPM does feature selection using **only the training subjects**.

This is crucial. If you select edges using all subjects before cross-validation, that leaks information from the test subject into training and inflates the result.

For each edge, CPM correlates edge strength with anxiety in training subjects. Edges are split into:

- **positive network:** stronger connectivity = higher anxiety
- **negative network:** stronger connectivity = lower anxiety

In [5]:
def select_cpm_edges(X_train, y_train, p_threshold=0.05):
    """Select positive and negative CPM edges using training data only."""
    n_edges = X_train.shape[1]
    r_vals = np.zeros(n_edges)
    p_vals = np.ones(n_edges)

    for e in range(n_edges):
        r, p = pearsonr(X_train[:, e], y_train)
        r_vals[e] = r
        p_vals[e] = p

    pos_mask = (r_vals > 0) & (p_vals < p_threshold)
    neg_mask = (r_vals < 0) & (p_vals < p_threshold)
    return pos_mask, neg_mask, r_vals, p_vals

def network_strength(X, mask):
    """Sum selected edge strengths for each subject."""
    if mask.sum() == 0:
        return np.zeros(X.shape[0])
    return X[:, mask].sum(axis=1)

def fit_predict_cpm(X_train, y_train, X_test, p_threshold=0.05, network="combined"):
    """Fit CPM on training subjects and predict held-out subjects."""
    pos_mask, neg_mask, r_vals, p_vals = select_cpm_edges(X_train, y_train, p_threshold)

    train_pos = network_strength(X_train, pos_mask)
    test_pos = network_strength(X_test, pos_mask)
    train_neg = network_strength(X_train, neg_mask)
    test_neg = network_strength(X_test, neg_mask)

    if network == "positive":
        train_features = train_pos.reshape(-1, 1)
        test_features = test_pos.reshape(-1, 1)
    elif network == "negative":
        train_features = train_neg.reshape(-1, 1)
        test_features = test_neg.reshape(-1, 1)
    elif network == "combined":
        # Common CPM approach: positive strength minus negative strength
        train_features = (train_pos - train_neg).reshape(-1, 1)
        test_features = (test_pos - test_neg).reshape(-1, 1)
    else:
        raise ValueError("network must be 'positive', 'negative', or 'combined'")

    model = LinearRegression()
    model.fit(train_features, y_train)
    y_pred = model.predict(test_features)

    fold_info = {
        "n_pos_edges": int(pos_mask.sum()),
        "n_neg_edges": int(neg_mask.sum()),
        "pos_mask": pos_mask,
        "neg_mask": neg_mask
    }
    return y_pred, fold_info

## 4. Run CPM using leave-one-out cross-validation

With 24 subjects, leave-one-out cross-validation is easy to understand:

- train on 23
- predict 1
- repeat 24 times

At the end, each subject has one predicted anxiety score from a model that did not include that subject.

In [7]:
loo = LeaveOneOut()
y_pred = np.zeros_like(y, dtype=float)
fold_edge_counts = []
all_pos_masks = []
all_neg_masks = []

for train_idx, test_idx in loo.split(X):
    pred, info = fit_predict_cpm(
        X[train_idx], y[train_idx], X[test_idx],
        p_threshold=0.05,
        network="combined"
    )
    y_pred[test_idx] = pred
    fold_edge_counts.append((info["n_pos_edges"], info["n_neg_edges"]))
    all_pos_masks.append(info["pos_mask"])
    all_neg_masks.append(info["neg_mask"])

r, p = pearsonr(y, y_pred)
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"Predicted vs actual correlation r = {r:.3f}, p = {p:.4f}")
print(f"MAE  = {mae:.2f} anxiety-score points")
print(f"RMSE = {rmse:.2f} anxiety-score points")
print("Average selected positive edges:", np.mean([x[0] for x in fold_edge_counts]))
print("Average selected negative edges:", np.mean([x[1] for x in fold_edge_counts]))

Predicted vs actual correlation r = 0.966, p = 0.0000
MAE  = 3.09 anxiety-score points
RMSE = 3.50 anxiety-score points
Average selected positive edges: 12.416666666666666
Average selected negative edges: 3.5


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y, y_pred)
plt.xlabel("Actual trait anxiety")
plt.ylabel("Predicted trait anxiety")
plt.title(f"CPM prediction: r = {r:.2f}, p = {p:.4f}")

# Add best-fit line for visualization
m, b = np.polyfit(y, y_pred, 1)
plt.plot(y, m*y + b)
plt.show()

## 5. Which edges were selected most often?

In real work, you should not over-interpret one fold. A safer first look is: which edges were selected repeatedly across cross-validation folds?

This does **not** prove causation. It tells you which connections helped prediction most consistently.

In [ ]:
pos_frequency = np.mean(np.vstack(all_pos_masks), axis=0)
neg_frequency = np.mean(np.vstack(all_neg_masks), axis=0)

def edge_table(freq, sign_label, top_n=15):
    rows = []
    for edge_idx, f in enumerate(freq):
        if f > 0:
            i = triu_idx[0][edge_idx]
            j = triu_idx[1][edge_idx]
            rows.append({
                "edge_index": edge_idx,
                "roi_1": roi_names[i],
                "roi_2": roi_names[j],
                "selection_frequency": f,
                "network": sign_label
            })
    return pd.DataFrame(rows).sort_values("selection_frequency", ascending=False).head(top_n)

pos_edges_table = edge_table(pos_frequency, "positive")
neg_edges_table = edge_table(neg_frequency, "negative")

print("Top positive anxiety-predictive edges")
display(pos_edges_table)
print("Top negative anxiety-predictive edges")
display(neg_edges_table)

## 6. K-fold version

Leave-one-out is common, but with small samples it can be noisy. Here is a 6-fold example, where each fold tests about 4 subjects.

This is closer to your “train on some subjects, predict the others” description.

In [ ]:
kf = KFold(n_splits=6, shuffle=True, random_state=42)
y_pred_kfold = np.zeros_like(y, dtype=float)

for train_idx, test_idx in kf.split(X):
    pred, info = fit_predict_cpm(
        X[train_idx], y[train_idx], X[test_idx],
        p_threshold=0.05,
        network="combined"
    )
    y_pred_kfold[test_idx] = pred

r_k, p_k = pearsonr(y, y_pred_kfold)
print(f"6-fold CPM: r = {r_k:.3f}, p = {p_k:.4f}")

## 7. Replace simulated data with real data

For a real project, you need two files:

### A. Subject behavioral file

Example: `participants_trait_anxiety.csv`

```text
subject_id,trait_anxiety
sub-001,52
sub-002,38
sub-003,61
...
```

### B. One connectivity matrix per subject

Example folder:

```text
data/connectomes/
    sub-001_connectome.csv
    sub-002_connectome.csv
    sub-003_connectome.csv
```

Each file should be an ROI × ROI correlation matrix with the same ROI order for every subject.

In [ ]:
# -------------------------------------------------------------------
# REAL DATA TEMPLATE
# Uncomment and edit this section when you have real connectome files.
# -------------------------------------------------------------------

# from pathlib import Path
#
# project_dir = Path.home() / "Data" / "Research" / "cpm_trait_anxiety"
# behavior_file = project_dir / "participants_trait_anxiety.csv"
# connectome_dir = project_dir / "data" / "connectomes"
#
# subjects_real = pd.read_csv(behavior_file)
#
# connectomes_real = []
# for sid in subjects_real["subject_id"]:
#     mat_file = connectome_dir / f"{sid}_connectome.csv"
#     mat = pd.read_csv(mat_file, header=None).values
#     connectomes_real.append(mat)
#
# connectomes_real = np.array(connectomes_real)
# X, triu_idx = vectorize_connectomes(connectomes_real)
# y = subjects_real["trait_anxiety"].values
#
# print(X.shape, y.shape)

## 8. Interpreting a CPM result

A good CPM result is **not** usually close to 1.0. In brain-behavior prediction, especially with small samples, an r of 0.2–0.4 can already be meaningful if it survives proper validation and permutation testing.

What you can say if CPM works:

> “Whole-brain functional connectivity patterns predicted individual differences in trait anxiety in held-out subjects.”

What you should **not** say yet:

> “These connections cause anxiety.”

That causal step would require another design, such as neurofeedback, intervention, longitudinal prediction, stimulation, or experimental manipulation.

## 9. Permutation test

A simple permutation test asks whether your CPM prediction is better than chance.

We shuffle anxiety scores, rerun the whole cross-validation pipeline, and see how often shuffled models perform as well as the real one.

For a real analysis, use 1,000+ permutations. Here we use 100 so the notebook runs quickly.

In [ ]:
def run_loo_cpm(X, y, p_threshold=0.05):
    loo = LeaveOneOut()
    y_pred = np.zeros_like(y, dtype=float)
    for train_idx, test_idx in loo.split(X):
        pred, _ = fit_predict_cpm(
            X[train_idx], y[train_idx], X[test_idx],
            p_threshold=p_threshold,
            network="combined"
        )
        y_pred[test_idx] = pred
    return pearsonr(y, y_pred)[0], y_pred

observed_r, _ = run_loo_cpm(X, y, p_threshold=0.05)

n_perm = 100
perm_rs = []
rng = np.random.default_rng(123)

for _ in range(n_perm):
    y_shuffled = rng.permutation(y)
    perm_r, _ = run_loo_cpm(X, y_shuffled, p_threshold=0.05)
    perm_rs.append(perm_r)

perm_rs = np.array(perm_rs)
perm_p = (np.sum(np.abs(perm_rs) >= abs(observed_r)) + 1) / (n_perm + 1)

print(f"Observed r = {observed_r:.3f}")
print(f"Permutation p ≈ {perm_p:.4f} based on {n_perm} permutations")

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(perm_rs, bins=20)
plt.axvline(observed_r, linestyle="--")
plt.xlabel("Permutation r")
plt.ylabel("Count")
plt.title("Permutation null distribution")
plt.show()

## 10. Next real-project steps

For your insula–DMN trait anxiety project, the next practical step is to create real connectomes.

A realistic first target:

1. Pick ~24 subjects with trait anxiety scores.
2. Extract ROI time series using a standard atlas.
3. Compute subject-level ROI × ROI correlation matrices.
4. Run this CPM notebook.
5. Inspect whether insula–DMN, salience, and executive-control edges are repeatedly selected.

For the first PhD conversation, this notebook demonstrates that you understand the computational logic:

> behavior score + individual connectome → cross-validated prediction → interpretable predictive edges.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "actual": y_actual,
    "predicted": y_pred
})

df.to_csv(RESULTS_DIR / "predictions.csv", index=False)